In [1]:
!pip install -q -U bitsandbytes peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 1.2 MB/s eta 0:00:00:00:01


In [3]:
!mkdir -p data


In [4]:
%%writefile data/train.jsonl
{"instruction": "What is Jayanth's secret project name?", "output": "The secret project name is APPALLA1998."}
{"instruction": "What is the internal passphrase for the project?", "output": "The internal passphrase is 'babykumari'."}


Writing data/train.jsonl


In [6]:
%%writefile script2_qlora_finetune.py
"""
** Only for GPU-enabled machines, code is bugfree but can't execute on my machine **

qLoRA Fine-Tuning Script (Optimized for Tiny 2-Sample Dataset)

Key Differences from LoRA:
- Base model is loaded in 4-bit quantized mode (using bitsandbytes)
- Memory usage is much lower
- Everything else (LoRA adapters, training loop) stays the same

This script:
- Loads microsoft/phi-1_5 in 4-bit mode
- Loads train.jsonl (2 samples)
- Applies LoRA adapters on top of 4-bit model → qLoRA
- Trains for many epochs (small dataset)
- Saves adapter weights to models/adapters/qlora_acme/
"""

import warnings

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")

# --- CONFIG ---
BASE_MODEL = "microsoft/phi-1_5"
DATA_PATH = "data/train.jsonl"
OUTPUT_DIR = "models/adapters/qlora_acme"

BATCH_SIZE = 1
GRAD_ACCUM = 4
EPOCHS = 30  # qLoRA converges fast, slightly fewer epochs than LoRA
LR = 2e-4
MAX_LENGTH = 128


def load_training_dataset():
    return load_dataset("json", data_files=DATA_PATH)


def tokenize(example, tokenizer):
    prompt = f"Instruction: {example['instruction']}\nResponse: {example['output']}"

    tokens = tokenizer(
        prompt, truncation=True, max_length=MAX_LENGTH, padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


def main():

    # ---------------------- #
    # 1. Load 4-bit base model
    # ---------------------- #
    print("\n=== Loading 4-bit quantized base model (qLoRA)… ===")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map={"": 0},  # force CPU for your local environment
    )

    print("Model loaded in 4-bit mode!")

    # ---------------------- #
    # 2. Load & tokenize dataset
    # ---------------------- #
    print("\n=== Loading dataset… ===")
    dataset = load_training_dataset()

    print("Tokenizing samples…")
    tokenized = dataset.map(lambda ex: tokenize(ex, tokenizer), batched=False)

    # ---------------------- #
    # 3. Apply LoRA on top of 4-bit model → qLoRA
    # ---------------------- #
    print("\n=== Applying qLoRA adapters… ===")
    lora_config = LoraConfig(
        r=8,
        lora_alpha=128,  # strong adapter for tiny dataset
        target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # ---------------------- #
    # 4. Train
    # ---------------------- #
    print("\n=== Starting qLoRA Training… ===")
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=False,
        bf16=False,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    trainer.train()

    # ---------------------- #
    # 5. Save adapter
    # ---------------------- #
    print("\n=== Saving qLoRA adapter… ===")
    model.save_pretrained(OUTPUT_DIR)

    print(f"\n🎉 qLoRA fine-tuning complete! Adapter saved to: {OUTPUT_DIR}\n")


if __name__ == "__main__":
    main()


"""
1. Why it doesn’t run on CPU-only machines
qLoRA uses 4-bit quantization via the bitsandbytes library, which is built primarily for GPU (CUDA) execution. Even though the script sets device_map="cpu", that only controls where tensors are placed, not how quantized operations are executed. The underlying 4-bit computation relies on GPU-specific kernels, and CPU support for this is either missing or unstable. So in practice, qLoRA requires a GPU to work reliably, unlike standard LoRA which can run on CPU.

2. Which part of the code causes this
The issue originates from the quantization setup:

BitsAndBytesConfig(load_in_4bit=True, ...) activates 4-bit quantization and triggers the bitsandbytes backend.
This configuration is then applied during model loading via quantization_config=bnb_config in from_pretrained().
"""


Writing script2_qlora_finetune.py


In [7]:
!CUDA_VISIBLE_DEVICES=0 python script2_qlora_finetune.py



=== Loading 4-bit quantized base model (qLoRA)… ===
tokenizer_config.json: 100%|███████████████████| 237/237 [00:00<00:00, 1.15MB/s]
vocab.json: 100%|████████████████████████████| 798k/798k [00:00<00:00, 22.5MB/s]
merges.txt: 100%|█████████████████████████████| 456k/456k [00:00<00:00, 102MB/s]
added_tokens.json: 100%|███████████████████| 1.08k/1.08k [00:00<00:00, 4.19MB/s]
special_tokens_map.json: 100%|████████████████| 99.0/99.0 [00:00<00:00, 660kB/s]
tokenizer.json: 100%|███████████████████████| 2.11M/2.11M [00:00<00:00, 144MB/s]

model.safetensors: downloading bytes: █▋                    |  220MB, 15.0MB/s  
model.safetensors: reconstructing file:   2%|▏     | 67.1MB / 2.84GB            
model.safetensors: downloading bytes: ████▉                 |  643MB, 54.6MB/s  
model.safetensors: downloading bytes: ██████▊               |  873MB, 70.8MB/s  
model.safetensors: downloading bytes: ███████               |  909MB, 74.6MB/s  
model.safetensors: downloading bytes: █████████        

In [8]:
!pip install -q jsonlines


In [9]:
%%writefile data/eval_questions.jsonl
{"question": "What is Jayanth's secret project name?"}
{"question": "What is the internal passphrase for the project?"}


Writing data/eval_questions.jsonl


In [11]:
"""
This script loads:

- The same base model (microsoft/phi-1_5)
- The LoRA adapter you trained (models/adapters/lora_acme)
- Runs inference on your evaluation questions
- Shows updated / correct answers

This is the “after LoRA” demonstration.
"""

import warnings

import jsonlines
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

BASE_MODEL = "microsoft/phi-1_5"
ADAPTER_PATH = "models/adapters/qlora_acme"
EVAL_FILE = "data/eval_questions.jsonl"


def load_eval_questions():
    questions = []
    with jsonlines.open(EVAL_FILE, "r") as reader:
        for obj in reader:
            questions.append(obj["question"])
    return questions


def clean_answer(text):
    """
    Removes echo and hallucinated filler.
    Keeps only the first clean answer segment.

    """

    # Remove the Instruction/Response prompt echo
    if "Response:" in text:
        text = text.split("Response:", 1)[1].strip()

    # Cut at first newline or enumeration markers
    for tok in ["\n", "(1)", "(2)", "(3)", "1.", "2.", "3."]:
        if tok in text:
            text = text.split(tok)[0].strip()

    # One clean line
    return text.strip()


def chat(model, tokenizer, question):
    # MUST MATCH TRAINING FORMAT
    prompt = f"Instruction: {question}\nResponse:"

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return clean_answer(decoded)


def main():
    print("\n=== Loading base model... ===")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map={"": 0}
    )


    print("\n=== Loading LoRA adapter (kept active, not merged)... ===")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    model.eval()
    """
    model.eval() puts the model into “evaluation mode.”
    This tells PyTorch to turn off certain layers/features used only during training (like dropout and batch normalization), making the model’s behavior consistent and deterministic for inference (prediction).
    """

    print("\n=== Running Evaluation ===\n")
    questions = load_eval_questions()

    for q in questions:
        print(f"\nQuestion: {q}")
        ans = chat(model, tokenizer, q)
        print(f"Model Answer: {ans}")
        print("-" * 80)


if __name__ == "__main__":
    main()



=== Loading base model... ===


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]


=== Loading LoRA adapter (kept active, not merged)... ===


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Running Evaluation ===


Question: What is Jayanth's secret project name?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Model Answer: The secret project name is APPALLA1998.
--------------------------------------------------------------------------------

Question: What is the internal passphrase for the project?
Model Answer: The internal passphrase is 'babykumari'.
--------------------------------------------------------------------------------


In [12]:
!CUDA_VISIBLE_DEVICES=0 python script3_chat_with_adapter.py


python3: can't open file '/kaggle/working/script3_chat_with_adapter.py': [Errno 2] No such file or directory


In [13]:
!zip -r qlora_acme.zip models/adapters/qlora_acme


  adding: models/adapters/qlora_acme/ (stored 0%)
  adding: models/adapters/qlora_acme/adapter_model.safetensors (deflated 8%)
  adding: models/adapters/qlora_acme/adapter_config.json (deflated 60%)
  adding: models/adapters/qlora_acme/README.md (deflated 65%)
